# 01 - Exploratory data analysis: PDBbind v2020 refined + CASF-2016

What this notebook does:

1. Loads the PDBbind v2020 refined-set index and the CASF-2016 core-set list.
2. Confirms the train/val/test split is leakage-free (no CASF-2016 PDB IDs in train or val).
3. Looks at the **pK distribution** (the regression target), affinity-type breakdown (Kd vs Ki vs IC50), release-year coverage, and resolution.

**Prerequisite:** run `python scripts/download_pdbbind.py` first to populate `data/raw/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from plb.data.pdbbind import find_default_paths, load_casf2016_coreset_ids, load_refined_index
from plb.data.splits import make_splits

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path("../data")
paths = find_default_paths(DATA_ROOT)
paths

In [ ]:
refined = load_refined_index(paths["refined_index"])
casf_ids = load_casf2016_coreset_ids(paths["casf_coreset"])
print(f"refined-set entries: {len(refined)}")
print(f"CASF-2016 core-set entries: {len(casf_ids)}")
refined.head()

In [ ]:
split = make_splits(refined["pdb_id"].tolist(), casf_ids, val_frac=0.1, seed=42)
split.sizes

### Sanity check: zero CASF-2016 leakage

If either of these prints a positive number, training would silently include test-set complexes.

In [ ]:
casf_set = set(casf_ids)
leaked_train = casf_set & set(split.train)
leaked_val = casf_set & set(split.val)
print(f"CASF leaks into train: {len(leaked_train)}")
print(f"CASF leaks into val:   {len(leaked_val)}")
assert not leaked_train and not leaked_val, "CASF-2016 leakage detected"

## pK distribution (the regression target)

PDBbind reports `-logKd/Ki` on a roughly 0-14 scale (higher = tighter binder). We plot the train and CASF-2016 distributions on the same axes so we can see whether the test set is representative.

In [ ]:
refined["subset"] = "train+val"
refined.loc[refined["pdb_id"].isin(casf_set), "subset"] = "CASF-2016 (test)"

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=refined,
    x="pK",
    hue="subset",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    ax=ax,
)
ax.set_title("pK distribution: train+val vs CASF-2016 (density-normalised)")
ax.set_xlabel("-log10(Kd / Ki / IC50)")
plt.tight_layout()

## Affinity-type breakdown

PDBbind mixes Kd, Ki and IC50 measurements. Knowing the proportions matters because IC50 is conditions-dependent and noisier than Kd / Ki.

In [ ]:
def affinity_kind(s: str) -> str:
    for k in ("Kd", "Ki", "IC50"):
        if s.startswith(k):
            return k
    return "other"


refined["affinity_kind"] = refined["affinity_raw"].map(affinity_kind)
refined.groupby(["subset", "affinity_kind"]).size().unstack(fill_value=0)

## Release year coverage

Useful for thinking about a future temporal split (Phase 6).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.histplot(data=refined, x="release_year", hue="subset", bins=30, ax=ax)
ax.set_title("PDBbind refined - release year by subset")
plt.tight_layout()

## Structural quality (resolution)

Should be tight: refined-set criteria require resolution <= 2.5A for X-ray structures.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=refined.dropna(subset=["resolution"]),
    x="resolution",
    hue="subset",
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    ax=ax,
)
ax.set_title("X-ray resolution (NMR entries excluded)")
ax.set_xlabel("resolution (Å)")
plt.tight_layout()

## Summary table

Numbers we'll cite in the README and report.md.

In [ ]:
summary = (
    refined.groupby("subset")
    .agg(
        n=("pdb_id", "count"),
        pK_mean=("pK", "mean"),
        pK_std=("pK", "std"),
        pK_min=("pK", "min"),
        pK_max=("pK", "max"),
        year_min=("release_year", "min"),
        year_max=("release_year", "max"),
    )
    .round(2)
)
summary